In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix,ConfusionMatrixDisplay

np.random.seed(42)
random.seed(42)
n_samples=300

data={
    'weather':np.random.choice(['sunny','cloudy','rainy'],n_samples),
    'soil':np.random.choice(['dry','moist','wet'],n_samples),
    'temperature':np.random.choice(['hot','mild','cool'],n_samples),
    'humidity':np.random.choice(['high','medium','low'],n_samples),
    'wind':np.random.choice(['strong','weak'],n_samples),
    'crop_type':np.random.choice(['corn','wheat','rice'],n_samples),
    'fertilizer':np.random.choice(['yes','no'],n_samples),
    'season':np.random.choice(['kharif','rabi','summer'],n_samples)
}
df=pd.DataFrame(data)

def irrigation_rule(row):
    if row['soil']=='dry' and row['weather']!='rainy':
      base='yes'
    elif row['soil']=='wet':
      base='no'
    else:
      base=random.choice(['yes','no'])

    if random.random()<0.2:
       return 'no' if base=='yes'else'yes'
    return base

df['irrigate']=df.apply(irrigation_rule,axis=1)

le=LabelEncoder()
for col in df.columns:
    df[col]=le.fit_transform(df[col])

X=df.drop('irrigate',axis=1)
y=df['irrigate']

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)
dt_model=DecisionTreeClassifier(random_state=42)
dt_model.fit(x_train,y_train)

y_pred_dt=dt_model.predict(x_test)

rf_model=RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    random_state=42
)
rf_model.fit(x_train,y_train)
y_pred_rf=rf_model.predict(x_test)

dt_acc=accuracy_score(y_test,y_pred_dt)
rf_acc=accuracy_score(y_test,y_pred_rf)

print("Decision Tree Accuracy:",dt_acc)
print("Random Forest Accuracy:",rf_acc)

cm_dt=confusion_matrix(y_test,y_pred_dt,labels=[0,1])
cm_rf=confusion_matrix(y_test,y_pred_rf,labels=[0,1])

print("\nDecision Tree Confusion Matrix:\n",cm_dt)
print("\nRandom Forest Confusion Matrix:\n",cm_rf)

ConfusionMatrixDisplay.from_predictions(y_test,y_pred_dt)
plt.title("Decision Tree")
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test,y_pred_rf)
plt.title("Random Forest")
plt.show()

dt_cv=cross_val_score(dt_model,X,y,cv=5)
rf_cv=cross_val_score(rf_model,X,y,cv=5)

print("\nCross validation Accuracy:")
print("Decision Tree:",dt_cv.mean())
print("Random Forest:",rf_cv.mean())



